# x4_pecuaria — Efetivo Bovino Municipal (PPM/IBGE)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** efetivo bovino, cabeças, municipal, anual  
**Conceito e Mecanismo Econômico:** Tamanho do rebanho bovino (cabeças) no município em 31 de dezembro. Representa a pressão de uso do solo pela pecuária de corte.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1_h2__ppm_bovinos__ibge__municipio_anual.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Explorar efetivo_bovino e composição produtiva anterior; rebanho é estoque, não produtividade.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município (141 municípios).
- **`ano`**: Ano da pesquisa.
- **`efetivo_bovino`**: Número total de cabeças de gado bovino.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1_h2__ppm_bovinos__ibge__municipio_anual.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x4_pecuaria`:
- **Atenção:** Explorar efetivo_bovino e composição produtiva anterior; rebanho é estoque, não produtividade.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7)
df['ano'] = pd.to_numeric(df['ano'], errors='coerce')
df['efetivo_bovino'] = pd.to_numeric(df['efetivo_bovino'], errors='coerce')

# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
df['codigo_ibge'] = df['codigo_ibge'].replace({'5101837': '5106240'})
df = df[df['codigo_ibge'].str.startswith('51')].copy()

# REGRA DO TXT: 'Explorar efetivo_bovino e composição produtiva anterior; rebanho é estoque, não produtividade.'
df = df.groupby(['codigo_ibge', 'ano'], as_index=False)['efetivo_bovino'].sum()
print(f"Municípios únicos: {df['codigo_ibge'].nunique()} (141 municípios)")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva do Efetivo Bovino (cabeças):")
print(df['efetivo_bovino'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['efetivo_bovino'], bins=30, kde=True, ax=axes[0], color='chocolate')
axes[0].set_title("Distribuição do Rebanho Bovino")

df_trend = df.groupby('ano')['efetivo_bovino'].sum().reset_index()
sns.lineplot(data=df_trend, x='ano', y='efetivo_bovino', marker='o', ax=axes[1], color='brown')
axes[1].set_title("Efetivo Bovino Total em MT (1974-2024)")
plt.tight_layout()
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df_painel = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)]
plt.figure(figsize=(10, 4))
sns.lineplot(data=df_painel, x='ano', y='efetivo_bovino', estimator='mean', color='brown')
plt.title("Rebanho Bovino Médio Municipal em MT (2001-2024)")
plt.ylabel("Cabeças")
plt.show()

## Fase 6: Feature Engineering — Efetivo Bovino Municipal (PPM/IBGE)

### 1. Step 1: Harmonização do Rebanho Bovino
- Posição anual em 31 de dezembro da Pesquisa Pecuária Municipal (PPM/IBGE).
- Salvaguarda para 141 municípios de Mato Grosso.

### 2. Step 2: Derivações
- **Transformação Logarítmica Defasada:** $\ln(1 + Rebanho_{t-1})$ estabiliza as ordens de grandeza entre cidades de 10 mil e 2 milhões de cabeças.
- **Taxa Anual de Crescimento do Rebanho:** $\Delta \ln(Rebanho_{i, t}) = \ln(Rebanho_{i, t}) - \ln(Rebanho_{i, t-1})$. Crescimentos acelerados de rebanho indicam pressão futura para abertura ou reforma de pastos.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Ordenação e Defasagens do Rebanho Bovino
df.sort_values(['codigo_ibge', 'ano'], inplace=True)

# efetivo_bovino: número total de cabeças de bovinos existentes no município em 31/dez
df['ln_bovino'] = np.log1p(df['efetivo_bovino'])
df['ln_bovino_lag1'] = df.groupby('codigo_ibge')['ln_bovino'].shift(1)

# 2. Variação Anual do Rebanho (dln_bovino ~ taxa de crescimento percentual)
df['dln_bovino'] = df.groupby('codigo_ibge')['ln_bovino'].diff()

painel_pec = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)].copy()

print(f"Painel Pecuário estruturado com {len(painel_pec)} linhas.")
print(painel_pec[['codigo_ibge', 'ano', 'efetivo_bovino', 'ln_bovino_lag1', 'dln_bovino']].head())